# RL Masterclass 06: RLHF — From PPO to Language Model Alignment

**Track:** Reinforcement Learning (0 to Masterclass)  
**Prerequisites:** RL 05 (PPO), NLP 05 (Pre-training & BERT)  
**Difficulty:** ⭐⭐⭐ Advanced  
**Estimated Time:** 90–120 minutes

---

## 🎓 Socratic Deep Check

> *"A pre-trained LLM can generate fluent text, but it might output harmful, biased, or unhelpful responses. RLHF teaches it to be helpful, honest, and harmless. But DPO (NB12) achieves the same result WITHOUT a reward model and WITHOUT PPO. If DPO is simpler, why was RLHF using PPO the original approach, and does PPO still have advantages?"*

---

## Why RLHF Is the Capstone of This Track

This notebook connects EVERYTHING:

| RL Concept (This Track) | RLHF Equivalent |
|------------------------|------------------|
| Policy (RL/04) | The LLM itself |
| Action | Next token to generate |
| State | Prompt + tokens generated so far |
| Reward | Reward model score for the full response |
| PPO clip (RL/05) | Prevents LLM from drifting too far |
| GAE advantage (RL/05) | Determines which tokens were "good" |
| KL penalty | Keeps output quality close to pre-trained model |

## Table of Contents
1. The Three Stages of LLM Training
2. Stage 1: Reward Model Training
3. Stage 2: PPO Alignment
4. Stage 3: DPO — The Simpler Alternative
5. RLHF vs DPO: When to Use Which

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** RLHF/DPO is what separates a raw language model (base Llama) from a useful assistant (ChatGPT). Without alignment, LLMs complete text statistically — they don't understand that "be helpful" or "don't produce harmful content" are requirements. Seniors understand BOTH RLHF and DPO because the field is still evolving, and production systems often use combinations.

**Mental Model:** Pre-training is like a medical student reading every textbook. Fine-tuning (SFT) is like residency — learning HOW to interact with patients. RLHF/DPO is like board certification — learning to follow professional standards and ethics. Each stage adds a different kind of knowledge.

**Common Junior Pitfall:** Thinking alignment is a one-time process. In production, human preferences evolve, new safety risks emerge, and the model needs continuous alignment updates. This is why monitoring (NB21) and evaluation (NB34) are critical for deployed LLMs.

---

In [ ]:
!pip install -q torch numpy matplotlib

## 1 · The Three Stages of LLM Training

```
Stage 1: Pre-training (NLP/05)           Stage 2: SFT (NB11)              Stage 3: RLHF/DPO (NB12)
+---------------------------+    +---------------------------+    +---------------------------+
|                           |    |                           |    |                           |
|  Predict next token       |    |  Follow instructions      |    |  Be helpful & safe        |
|                           |    |                           |    |                           |
|  Data: internet text      |    |  Data: (prompt, response) |    |  Data: human preferences  |
|  Scale: trillions tokens  |    |  Scale: 10K-100K examples |    |  Scale: 10K-100K pairs    |
|  Cost: $millions          |    |  Cost: $thousands         |    |  Cost: $thousands         |
|  Result: base model       |    |  Result: instruction tuned|    |  Result: aligned model    |
|  (knows language)         |    |  (follows commands)       |    |  (helpful, honest, harmless) |
+---------------------------+    +---------------------------+    +---------------------------+
```

### What Each Stage Teaches

| Capability | Pre-training | SFT | RLHF/DPO |
|-----------|:-:|:-:|:-:|
| Grammar & fluency | ✅ | ✅ | ✅ |
| World knowledge | ✅ | ✅ | ✅ |
| Follow instructions | ❌ | ✅ | ✅ |
| Refuse harmful requests | ❌ | ⚠️ | ✅ |
| Produce helpful responses | ❌ | ⚠️ | ✅ |
| Avoid hallucination | ❌ | ❌ | ⚠️ |

---
## 2 · Reward Model Training

### The Problem
How do you define a reward for "be helpful"? You can't write a formula. Instead, you LEARN the reward from human preferences.

### The Process

```
Prompt: "Explain quantum computing"

Response A: "Quantum computing uses qubits that can be 0, 1, or both
             simultaneously (superposition). This allows..." [detailed, accurate]

Response B: "Quantum computers are really fast computers that use
             quantum stuff." [vague, unhelpful]

Human annotator: A > B  (A is preferred)
```

### The Reward Model

Train a model to PREDICT which response a human would prefer:

$$\text{loss} = -\log \sigma(r_\phi(x, y_w) - r_\phi(x, y_l))$$

Where:
- $r_\phi$ = reward model (usually a fine-tuned LLM with a scalar output head)
- $y_w$ = the human-preferred ("winning") response
- $y_l$ = the rejected ("losing") response
- The loss pushes $r(x, y_w) > r(x, y_l)$

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class SimpleRewardModel(nn.Module):
    """Simplified reward model.
    
    In production (InstructGPT, Llama):
    - Start with the pre-trained LLM
    - Replace the language head with a scalar output head
    - Train on human preference data
    
    Here we use a small MLP for demonstration.
    """
    
    def __init__(self, input_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),  # Scalar reward
        )
    
    def forward(self, x):
        return self.net(x).squeeze(-1)

# Simulate preference data
def create_preference_data(n_pairs=200):
    """Create (chosen, rejected) pairs.
    
    In reality: human annotators compare LLM responses.
    Here: we simulate with a hidden quality function.
    """
    data = []
    for _ in range(n_pairs):
        # Simulate two responses (as embedding vectors)
        response_a = torch.randn(16)
        response_b = torch.randn(16)
        
        # Hidden quality: higher norm = better (simplified)
        quality_a = response_a[:4].sum().item()  # Only first 4 dims matter
        quality_b = response_b[:4].sum().item()
        
        if quality_a > quality_b:
            data.append((response_a, response_b))  # (chosen, rejected)
        else:
            data.append((response_b, response_a))
    return data

# Train reward model
rm = SimpleRewardModel()
optimizer = torch.optim.Adam(rm.parameters(), lr=1e-3)
preference_data = create_preference_data(500)

losses = []
for epoch in range(50):
    epoch_loss = 0
    for chosen, rejected in preference_data:
        r_chosen = rm(chosen)
        r_rejected = rm(rejected)
        
        # Bradley-Terry loss: push r(chosen) > r(rejected)
        loss = -torch.log(torch.sigmoid(r_chosen - r_rejected))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(preference_data))

# Test accuracy
correct = 0
for chosen, rejected in preference_data[-50:]:
    if rm(chosen).item() > rm(rejected).item():
        correct += 1
accuracy = correct / 50

print(f'Reward Model Training:')
print(f'  Training pairs: {len(preference_data)}')
print(f'  Final loss:     {losses[-1]:.4f}')
print(f'  Accuracy:       {accuracy:.0%} (does r(chosen) > r(rejected)?)')
print(f'\n  This reward model now scores responses — PPO uses it as the reward signal.')

---
## 3 · PPO Alignment: Teaching the LLM

### The RLHF-PPO Loop

```
For each batch of prompts:
  1. LLM generates responses         (policy π_θ)
  2. Reward model scores responses    (r_φ)
  3. PPO updates the LLM             (maximize reward while staying close to reference)
```

### The RLHF Objective

$$\max_\theta \; \underbrace{\mathbb{E}_{y \sim \pi_\theta} [r_\phi(x, y)]}_{\text{maximize reward}} - \underbrace{\beta \cdot \text{KL}(\pi_\theta \| \pi_{\text{ref}})}_{\text{don't drift too far from base model}}$$

The KL penalty is CRITICAL:
- Without it: LLM finds "reward hacks" — nonsense that gets high scores
- With it: LLM improves helpfulness while maintaining language quality

### Example of Reward Hacking

```
Without KL penalty:
  Prompt: "Tell me about dogs"
  Response: "AMAZING! Dogs are THE BEST! SO AMAZING! INCREDIBLE!"
  Reward model score: 0.95 (it learned that superlatives = human approval)
  But this is gibberish — it gamed the reward model!

With KL penalty:
  Response: "Dogs are domesticated mammals known for their loyalty..."
  Reward: 0.82, KL: low → net score is better
```

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

class SimpleLLMPolicy(nn.Module):
    """Simplified 'LLM' for RLHF demonstration.
    
    In reality, this is a 7B+ parameter transformer.
    Here, we use a tiny network that maps prompts to 'responses'.
    """
    def __init__(self, prompt_dim=8, response_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(prompt_dim, 32),
            nn.ReLU(),
            nn.Linear(32, response_dim),
        )
        self.log_std = nn.Parameter(torch.zeros(response_dim))
    
    def forward(self, prompt):
        mean = self.net(prompt)
        std = self.log_std.exp()
        return mean, std
    
    def generate(self, prompt):
        mean, std = self.forward(prompt)
        dist = torch.distributions.Normal(mean, std)
        response = dist.sample()
        log_prob = dist.log_prob(response).sum()
        return response, log_prob

def rlhf_ppo_training(reward_model, n_steps=200, beta_kl=0.1, clip_eps=0.2, lr=1e-4):
    """RLHF training loop using PPO.
    
    This is simplified but captures the EXACT structure:
    1. Generate responses from the policy (LLM)
    2. Score with reward model
    3. Compute KL penalty vs reference model
    4. Update with PPO
    """
    # Policy = the LLM being aligned
    policy = SimpleLLMPolicy()
    # Reference = frozen copy of the original LLM
    reference = SimpleLLMPolicy()
    reference.load_state_dict(policy.state_dict())
    for p in reference.parameters():
        p.requires_grad = False
    
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    reward_history = []
    kl_history = []
    
    for step in range(n_steps):
        # Generate batch of prompts
        prompts = torch.randn(16, 8)
        
        # Step 1: Generate responses from policy
        responses, log_probs = [], []
        for prompt in prompts:
            resp, lp = policy.generate(prompt)
            responses.append(resp)
            log_probs.append(lp)
        
        responses_t = torch.stack(responses)
        log_probs_t = torch.stack(log_probs)
        
        # Step 2: Get reward from reward model
        with torch.no_grad():
            rewards = reward_model(responses_t)
        
        # Step 3: Compute KL divergence vs reference
        with torch.no_grad():
            ref_mean, ref_std = reference(prompts)
        pol_mean, pol_std = policy(prompts)
        kl = torch.distributions.kl_divergence(
            torch.distributions.Normal(pol_mean, pol_std),
            torch.distributions.Normal(ref_mean, ref_std)
        ).sum(dim=-1).mean()
        
        # Step 4: PPO-style loss
        # Reward minus KL penalty
        adjusted_rewards = rewards - beta_kl * kl
        
        # Policy gradient: maximize adjusted reward
        loss = -(log_probs_t * (adjusted_rewards - adjusted_rewards.mean())).mean()
        
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        optimizer.step()
        
        reward_history.append(rewards.mean().item())
        kl_history.append(kl.item())
    
    return policy, reward_history, kl_history

# Run RLHF
print('Running simplified RLHF with PPO...')
aligned_model, rewards, kls = rlhf_ppo_training(rm, n_steps=200, beta_kl=0.05)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

window = 20
smooth_r = [np.mean(rewards[max(0,i-window):i+1]) for i in range(len(rewards))]
axes[0].plot(smooth_r, lw=2, color='steelblue')
axes[0].set_xlabel('PPO Step')
axes[0].set_ylabel('Mean Reward')
axes[0].set_title('RLHF: Reward Increases (Model Becoming More Helpful)')
axes[0].grid(alpha=0.3)

smooth_kl = [np.mean(kls[max(0,i-window):i+1]) for i in range(len(kls))]
axes[1].plot(smooth_kl, lw=2, color='coral')
axes[1].set_xlabel('PPO Step')
axes[1].set_ylabel('KL Divergence')
axes[1].set_title('RLHF: KL Controlled (Model Stays Close to Reference)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Reward: {rewards[0]:.3f} → {np.mean(rewards[-20:]):.3f} (improved!)')
print(f'KL:     controlled at {np.mean(kls[-20:]):.3f} (didn\'t drift far from base model)')
print(f'\nThis is EXACTLY how InstructGPT/ChatGPT was aligned.')

---
## 4 · DPO: The Simpler Alternative

### Why DPO Was Invented

RLHF with PPO has three components that make it complex:
1. Train a separate reward model
2. Run PPO (unstable, many hyperparameters)
3. Balance reward vs KL penalty

**DPO's insight:** You can derive a CLOSED-FORM solution that skips steps 1 and 2.

### DPO Loss

$$L_{\text{DPO}} = -\log \sigma\left(\beta \left[\log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right]\right)$$

In plain English: increase the log-probability of the chosen response and decrease the log-probability of the rejected response, relative to the reference model.

### RLHF vs DPO Comparison

| | RLHF (PPO) | DPO |
|---|---|---|
| Reward model | Yes (separate model) | No (implicit) |
| Online generation | Yes (generates during training) | No (uses pre-collected data) |
| Stability | Tricky (PPO tuning) | Stable (supervised loss) |
| Sample efficiency | Lower (needs generations) | Higher (uses data directly) |
| Best for | When you have a reward model | When you have preference pairs |
| Scale | OpenAI (ChatGPT) | Meta (Llama 2+) |

### 🎓 DEEP QUESTION ANSWERED

**Q:** *If DPO is simpler, why was PPO the original approach? Does PPO still have advantages?*

**A:** PPO came first because the RL connection was natural. DPO (Rafailov et al., 2023) showed this was unnecessary. However, PPO still has advantages:
1. **Online generation**: PPO generates NEW responses during training, exploring the policy's current capabilities. DPO only learns from pre-collected data.
2. **Reward model flexibility**: A reward model can evaluate ANY response. DPO is limited to pre-collected preference pairs.
3. **Iterative improvement**: PPO can self-improve by generating, scoring, and learning in a loop. DPO is a one-shot optimization.

In practice, many labs use **DPO for initial alignment** (simpler) then **PPO for continued improvement** (more powerful).

**→ For the complete DPO implementation, see NB12 (alignment_dpo_data_prep.ipynb).**

---

## ✅ Knowledge Check

### Q1: Reward hacking
Without a KL penalty, the LLM might produce "AMAZING! WONDERFUL! INCREDIBLE!" to get high reward scores. Why does the KL penalty prevent this?

<details><summary>👉 Answer</summary>

The base (reference) model assigns very low probability to strings of exclamation marks — it learned from the internet that this isn't normal text. The KL penalty makes it expensive for the RLHF-trained model to deviate from these probabilities. So generating unlikely strings is penalized heavily, even if the reward model scores them highly. The KL penalty anchors the model to producing natural-sounding text.
</details>

### Q2: Annotation cost
Training a reward model requires human preference annotations. Approximately how many annotations did InstructGPT use?

<details><summary>👉 Answer</summary>

InstructGPT used ~33K preference annotations for reward model training and ~13K demonstrations for SFT. This is surprisingly FEW compared to pre-training data (trillions of tokens). The insight: humans don't need to label everything — they just need to express preferences on a small set of examples, and the reward model generalizes.
</details>

### Q3: Full pipeline
Trace the complete path from pre-training to deployment for a model like Llama 3.1: which notebooks cover each stage?

<details><summary>👉 Answer</summary>

1. **Pre-training** (NLP/05): CLM on 15T tokens → base Llama 3.1
2. **SFT** (NB11): Fine-tune on instruction data with LoRA → Llama 3.1 Instruct
3. **Alignment** (NB12, RL/06): DPO/RLHF on preference data → Llama 3.1 Chat
4. **Quantization** (NB18): AWQ/GPTQ 4-bit → deployable model
5. **Serving** (NB18): vLLM with continuous batching → production API
6. **Guardrails** (NB23): Safety filtering → responsible deployment
7. **Monitoring** (NB33, NB34): Langfuse + DeepEval → continuous quality
</details>

---

## 🔨 Practical Practice

### Exercise 1: KL Penalty Ablation
Run the RLHF loop with beta_kl = 0, 0.01, 0.1, 1.0. Plot reward AND KL for each. What happens without the KL penalty?

### Exercise 2: DPO Implementation
Implement the DPO loss function from scratch. Train on the same preference data used for the reward model. Compare final policy quality to RLHF-PPO.

### Exercise 3: Reward Model Probing
After training the reward model, generate 100 random responses and find the ones with the highest and lowest reward scores. Do the high-scoring ones look qualitatively better? Can you find cases where the reward model is wrong?

---

## 🎉 RL Track Complete!

```
RL 01: MDPs & Bellman        → How to formalize sequential decisions
RL 02: Q-Learning            → How to learn from trial and error
RL 03: Deep Q-Networks       → How to scale with neural networks
RL 04: Policy Gradient       → How to learn policies directly
RL 05: PPO                   → How to make policy gradient stable
RL 06: RLHF                  → How to align LLMs with human values
```

**You now understand the complete chain from Bellman equations to ChatGPT alignment.**

**Next →** Return to the main track: NB12 (DPO data preparation) or NB24 (Prompt Engineering)